# 2D to 3D lifting with MotionBERT (Colab)

Takes the RTMPose keypoints exported by `src/pose/export_h36m.py` (file `h36m_2d.npz`, key `motionbert_input`)
and lifts them to 3D with the pretrained MotionBERT-Lite 3D-pose model. Nothing is trained.

**Use a GPU runtime** (Runtime > Change runtime type). Written from the MotionBERT repository's `infer_wild.py`; I could not run it
(the weights are not reachable from where it was written), so if a cell fails, copy the error back.
Output: `lift3d_<clip>.npz` with `pose3d_conf` and `pose3d_noconf` (confidence set to 1), each (T, 17, 3), in normalised units (not metres).
H36M joints: 0 pelvis, 1 R hip, 2 R knee, 3 R ankle, 4 L hip, 5 L knee, 6 L ankle, 7 spine, 8 thorax, 9 neck/nose, 10 head, 11 L shoulder, 12 L elbow, 13 L wrist, 14 R shoulder, 15 R elbow, 16 R wrist.

In [ ]:
!git clone -q https://github.com/Walter0807/MotionBERT.git
%cd MotionBERT
!pip install -q easydict pyyaml tqdm imageio imageio-ffmpeg huggingface_hub

In [ ]:
# Download the checkpoint from the Hugging Face mirror
from huggingface_hub import hf_hub_download, list_repo_files
print([f for f in list_repo_files('walterzhu/MotionBERT') if 'pose3d' in f])   # check the path below is in this list
CKPT = 'checkpoint/pose3d/FT_MB_lite_MB_ft_h36m_global_lite/best_epoch.bin'
hf_hub_download('walterzhu/MotionBERT', CKPT, local_dir='.')

In [ ]:
# Build the model, as in infer_wild.py. Always wrap in DataParallel: the checkpoint keys carry the "module." prefix.
import numpy as np, torch, torch.nn as nn
from lib.utils.tools import get_config
from lib.utils.learning import load_backbone
from lib.utils.utils_data import flip_data

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
args = get_config('configs/pose3d/MB_ft_h36m_global_lite.yaml')
model = nn.DataParallel(load_backbone(args))
if dev == 'cuda':
    model = model.cuda()
ck = torch.load(CKPT, map_location=lambda s, l: s)
model.load_state_dict(ck['model_pos'], strict=True)
model.eval()
print('model ready on', dev)

def lift(x, use_conf=True, clip_len=243, flip=True):
    """x: (T, 17, 3) = normalised x, y and confidence. Returns (T, 17, 3).
    use_conf=False keeps 3 channels but sets confidence to 1: this checkpoint's first layer needs 3 channels."""
    out = []
    with torch.no_grad():
        for s in range(0, len(x), clip_len):                       # non-overlapping clips, as in the original script
            b = torch.tensor(x[s:s + clip_len], dtype=torch.float32)[None].to(dev)   # (1, T, 17, 3)
            if not use_conf:
                b = b.clone(); b[..., 2] = 1.0
            p = model(b)
            if flip:
                p2 = flip_data(model(flip_data(b)))
                p = (p + p2) / 2.0
            p[:, 0, 0, 2] = 0                                      # root depth of the first frame set to 0 (rootrel=False case)
            out.append(p[0].cpu().numpy())
    return np.concatenate(out, axis=0)

In [ ]:
# Upload the h36m_2d.npz of each clip (one at a time), lift, and download the result.
from google.colab import files
clip = 'deadlift_set1'            # MUST match the file you upload: deadlift_set2 (1060 frames) or deadlift_set1 (1091 frames)
up = files.upload()               # choose outputs/<clip>_rtm/h36m_2d.npz
d = np.load(list(up)[0])
x = d['motionbert_input']
EXPECTED = {'deadlift_set2': 1060, 'deadlift_set1': 1091}   # add new clips here
print('clip:', clip, '| frames:', x.shape[0])
if clip in EXPECTED:
    assert x.shape[0] == EXPECTED[clip], f"{x.shape[0]} frames does not match {clip}: wrong clip name or wrong file"
p_conf = lift(x, use_conf=True)
p_noconf = lift(x, use_conf=False)
print(p_conf.shape, np.abs(p_conf - p_noconf).mean())
np.savez(f'lift3d_{clip}.npz', pose3d_conf=p_conf, pose3d_noconf=p_noconf)
files.download(f'lift3d_{clip}.npz')


## Oracle masking experiment (optional, run after the cell above for the same clip)
Marks the left knee as missing in a window of +-K frames around each frame where I marked it hidden (`hidden_frames`, stored in
`h36m_2d.npz` by `export_h36m.py`), then lifts again. This uses my labels to say where the knee is lost, so it is an upper bound,
not a usable method. How MotionBERT treats a joint with confidence 0 is NOT verified: the `conf0` and `zero` variants are two guesses
at "missing"; `interp` replaces the knee by interpolation from frames outside the window. Evaluate locally with `src/metrics/eval_masking.py`.

In [ ]:
hid = d['hidden_frames']
print('clip:', clip, '| hidden-knee frames:', hid.tolist())
assert len(hid) > 0, 'no hidden_frames in this file: re-export it with export_h36m.py (needs the labels)'
KNEE = 5   # H36M left knee (the side evaluated)

def window(T, hid, K):
    m = np.zeros(T, bool)
    for f in hid:
        m[max(0, f - K):min(T, f + K + 1)] = True
    return m

def apply(x, m, variant):
    y = x.copy()
    if variant == 'conf0':
        y[m, KNEE, 2] = 0.0
    elif variant == 'zero':
        y[m, KNEE, :] = 0.0
    elif variant == 'interp':
        t = np.arange(len(x))
        for c in (0, 1):
            y[m, KNEE, c] = np.interp(t[m], t[~m], x[~m, KNEE, c])
    return y

out = {'base': lift(x)}
for K in (5, 15):
    m = window(len(x), hid, K)
    for v in ('conf0', 'zero', 'interp'):
        out[f'{v}_K{K}'] = lift(apply(x, m, v))
    print('K =', K, '| masked frames:', int(m.sum()))
np.savez(f'lift3d_{clip}_masked.npz', **out)
files.download(f'lift3d_{clip}_masked.npz')